[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/block5.ipynb)

# Block 5: one dial, three answers

Nineteen minutes, together. Every answer is written. Read, run, look.

The model in this notebook is the one you built by hand in block 3: it counts
which word followed which, and then guesses. It is small enough to read, and
the dial on it is the same dial as the one on the tools you will use at work.

## 1. The text, and the counts

Fifty sentences. Everything below is counted from them and nothing is
downloaded.

In [ ]:
SENTENCES = [
    "the cat sat on the mat",
    "the cat sat on the mat",
    "the cat sat on the mat",
    "the cat sat on the mat",
    "the cat sat on the mat",
    "the black cat sat on the mat",
    "the old cat sat on the mat",
    "my cat sat on the mat",
    "the cat sat on the chair",
    "the cat sat on the chair",
    "the dog sat on the floor",
    "the dog sat on the floor",
    "the small dog sat on the sofa",
    "the cat slept on the sofa",
    "the dog slept on the floor",
    "the bird sang in the morning",
    "the bird flew over the house",
    "the children played in the garden",
    "the children read a book in the library",
    "the teacher wrote a word on the board",
    "the teacher asked a question about the book",
    "the student answered the question with a smile",
    "the student read the book about the sea",
    "a man walked along the beach in the evening",
    "a woman swam in the sea before breakfast",
    "the sun rose over the sea",
    "the sun set behind the mountains",
    "the rain fell on the roof all night",
    "the wind blew through the trees",
    "the train arrived at the station on time",
    "the train left the station in the morning",
    "the bus stopped near the market",
    "the market opened early in the morning",
    "the coffee was hot and the bread was fresh",
    "the summer school met in the old town",
    "the students walked to the lecture hall",
    "the lecture began with a simple question",
    "the professor drew a chart on the board",
    "the chart showed a rising line",
    "the model guessed the next word",
    "the model learned from the data",
    "the data came from many books",
    "the books stood on the shelf",
    "the shelf held many old books",
    "the library was quiet in the afternoon",
    "the afternoon passed quickly by the sea",
    "the evening brought a cool wind from the sea",
    "the city was calm at night",
    "the night was clear and the stars were bright",
    "the morning began with strong coffee",
]

pairs = {}
for sentence in SENTENCES:
    words = sentence.split()
    for first, second in zip(words, words[1:]):
        pairs.setdefault(first, {}).setdefault(second, 0)
        pairs[first][second] += 1

print("words in the text:", sum(len(s.split()) for s in SENTENCES))
print("words that ever start a pair:", len(pairs))
print()
print("after 'cat':", pairs["cat"])
print("after 'the':", pairs["the"])

## 2. The dial

Every count is raised to the power one over the dial setting, and the results
are divided by their total so they add to one.

Low settings make the commonest word take almost everything. High settings
share the weight out. The counts never change: the dial only changes how they
are read.

In [ ]:
def shares(word, setting):
    counts = pairs[word]
    weighted = {w: n ** (1.0 / setting) for w, n in counts.items()}
    total = sum(weighted.values())
    return {w: v / total for w, v in sorted(weighted.items(),
                                            key=lambda kv: -kv[1])}

for setting in (0.4, 1.0, 2.0):
    line = ", ".join("%s %.0f%%" % (w, 100 * s)
                     for w, s in shares("cat", setting).items())
    print("dial %.1f:  %s" % (setting, line))

## 3. Three answers from one model

The same counts, the same starting word, three settings. The only difference
between these three lines is the dial.

A line can stop early. That happens when the walk reaches a word the text never
put anything after.

In [ ]:
import random


def write(start, setting, length=8, seed=0):
    rng = random.Random(seed)
    word, out = start, [start]
    for _ in range(length):
        if word not in pairs:
            break
        weights = shares(word, setting)
        word = rng.choices(list(weights), weights=list(weights.values()))[0]
        out.append(word)
    return " ".join(out)


for setting in (0.4, 1.0, 2.0):
    print("dial %.1f:  %s" % (setting, write("the", setting)))

## 4. Your turn

Change one thing at a time and run the cell again.

1. Raise the setting to 5.0. At what point does the line stop being a sentence?
2. Drop it to 0.1. Run it three times. What is the same every time, and why?
3. Start from `cat`.

Write down the setting at which the answers stopped being usable. That number
is why the dial is on the tool.

In [ ]:
for setting in (0.1, 0.4, 1.0, 2.0, 5.0):
    print("dial %.1f:  %s" % (setting, write("the", setting, seed=1)))

## 5. How often it is caught out

Forty sentences to count from, ten kept back. Two models: one that always
answers with the commonest word in the text, and one that looks back a single
word. Both are asked to guess every next word.

The number to watch is what happens on the ten sentences the counting never
saw.

In [ ]:
train, test = SENTENCES[:40], SENTENCES[40:]

seen_pairs, freq = {}, {}
for sentence in train:
    words = sentence.split()
    for word in words:
        freq[word] = freq.get(word, 0) + 1
    for first, second in zip(words, words[1:]):
        seen_pairs.setdefault(first, {}).setdefault(second, 0)
        seen_pairs[first][second] += 1

commonest = max(sorted(freq), key=lambda w: freq[w])


def guess_rate(sentences):
    none = back = total = 0
    for sentence in sentences:
        words = sentence.split()
        for first, true in zip(words, words[1:]):
            total += 1
            none += (commonest == true)
            after = seen_pairs.get(first, {})
            pick = (max(sorted(after), key=lambda w: after[w]) if after
                    else commonest)
            back += (pick == true)
    return 100 * none / total, 100 * back / total, total


for label, group in (("counted from", train), ("kept back", test)):
    no_context, one_back, total = guess_rate(group)
    print("%-14s %4d next words:  no context %2.0f%%   one word back %2.0f%%"
          % (label, total, no_context, one_back))

## What to take away

The dial is free, it is on every tool you will open, and it decides whether
you get the same answer twice.

And the second number in the last cell is the one to believe. A model measured
on the text it learned from is measuring its memory.